# Ordered Logistic Regression Results: Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2 certified dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and covers ordered logistic regression results for adoption predictors of indigenous and modern knowledge in rangeland management in Northern Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their field and column `@id`s for further analysis.

We'll print out record set `@id`s, their descriptions, and list fields and columns for each. All identifiers strictly use `@id` as required.

In [ ]:
# List all record sets, their fields, and columns using their @id
from pprint import pprint

record_sets_info = []  # For later reference
print('Available Record Sets:')
for record_set in dataset.record_sets:
    print(f"- Record set @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', '(no name)')}")
    print(f"  Description: {record_set.get('description', '(no description)')}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        # Single field object case
        fields = [fields]
    print(f"  Fields:")
    fields_ids = []
    for f in fields:
        if isinstance(f, dict):
            field_id = f.get('@id') or f.get('field', {}).get('@id')
        else:  # It's just the @id string
            field_id = f
        print(f"    - {field_id}")
        fields_ids.append(field_id)
    columns = record_set.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    print(f"  Columns:")
    columns_ids = []
    for c in columns:
        if isinstance(c, dict):
            column_id = c.get('@id') or c.get('column', {}).get('@id')
        else:
            column_id = c
        print(f"    - {column_id}")
        columns_ids.append(column_id)
    record_sets_info.append({
        'record_set_id': record_set['@id'],
        'fields': fields_ids,
        'columns': columns_ids
    })
    print('-' * 50)

# Pick the first record set for demonstration
if record_sets_info:
    main_record_set_id = record_sets_info[0]['record_set_id']
    main_fields = record_sets_info[0]['fields']
    main_columns = record_sets_info[0]['columns']
else:
    main_record_set_id = None
    main_fields = []
    main_columns = []

## 3. Data Extraction
Load data from a chosen record set into a DataFrame for analysis. We'll use record set and field/column `@id`s from the overview above. If there are multiple record sets, we'll load all as DataFrames and show the main one.

In [ ]:
# Extract data from all record sets using their @id
dataframes = {}
loaded_ids = []

for rsi in record_sets_info:
    rsid = rsi['record_set_id']
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        loaded_ids.append(rsid)
        print(f"Loaded DataFrame for record_set @id: {rsid} (rows: {len(df)})")
        print(f"Columns: {df.columns.tolist()}")
    except Exception as ex:
        print(f"Could not load record_set {rsid}: {ex}")
        continue

if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nPreview of main record set: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No main record set loaded for preview.")

## 4. Exploratory Data Analysis (EDA)
Apply initial processing: filter out rows based on a numeric column, normalize numeric data, and group by a key attribute.
- Use the column and field `@id`s printed above. 
- Adapt this section according to what is available: if no numeric data is present, show a demo with available fields/columns.

In [ ]:
# EDA: Filter, normalize and group data using `@id`s
import numpy as np

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id].copy()
    
    # Guess a numeric field/column @id for demonstration
    # Try to find a numeric-looking column
    numeric_col_candidates = [c for c in df.columns if c.lower().startswith('log_likelihood') or df[c].dtype in [np.float64, np.int64]]
    if not numeric_col_candidates:
        # fallback to first float/integer column
        numeric_col_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_col_candidates:
        numeric_field_id = numeric_col_candidates[0]  # Use first candidate
        print(f"Using column for numeric analysis: {numeric_field_id}")
        # Remove rows with missing/NaN values in the numeric column
        df_nonan = df.dropna(subset=[numeric_field_id])
        threshold = df_nonan[numeric_field_id].mean()  # Use mean as threshold demo
        
        filtered_df = df_nonan[df_nonan[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group by a categorical field if any exists
        # Try to pick a string/object type column not equal to the numeric
        group_fields = [c for c in df.columns if c != numeric_field_id and df[c].dtype == object]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical column found to group by.")
    else:
        print("No numeric column found for analysis in main record set.")
else:
    print("No data to analyze from main record set.")

## 5. Visualization
Visualize distributions or relationships in the data. We will create a histogram for the selected numeric field and a boxplot if a grouping field is available.

In [ ]:
import matplotlib.pyplot as plt

if main_record_set_id and main_record_set_id in dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,5))
    df = dataframes[main_record_set_id].dropna(subset=[numeric_field_id])
    plt.hist(df[numeric_field_id], bins=30, color='steelblue', edgecolor='k')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    # Boxplot by group_field if available
    if 'group_field' in locals() and group_field in df:
        plt.figure(figsize=(10,6))
        df.boxplot(column=numeric_field_id, by=group_field)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.suptitle("")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field)
        plt.show()
else:
    print("No numeric or group field found for visualization.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using the `mlcroissant` library, loaded record-level data by their `@id`, and performed initial data processing and visualization. For in-depth analysis, you can further explore specific fields and model outputs as defined by their Croissant schema identifiers.

- Always use `@id` values to refer to record sets and fields for maximal reproducibility with Croissant datasets.
- For documentation and full variable definitions, see the dataset's [Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) or consult `dataset.metadata`.

_This notebook can be adapted for any Croissant-structured dataset by following the template steps above._